# Car fleet and emissions by mandate scenario

New registrations from `zevm.scenarios` → cars on the road (parc) → vehicle miles → fuel → tailpipe CO2e, to 2035.
A port of archive notebook 01 (Parts 1–3). Method and its differences from the notebook are at the top of `zevm/fleet.py`;
every assumption, with its source, is in `params/fleet.yaml`.

| assumption | where |
|---|---|
| lifespan 18 years, retirements spread over 10 | `retirement` |
| split of non-ZEVs into petrol / diesel / hybrid | `registrations.nonzev_split` |
| annual mileage per fuel (MOT data), held flat | `mileage` |
| new cars' fuel use vs the base-year fleet | `fuel_use.new_vs_fleet` |
| DESNZ CO2e factors, scaling to NAEI | `emissions` |

To try a different assumption, write a small file with `extends: fleet.yaml` and load it with `fleet.load_params("my_file")`.

In [ ]:
import pandas as pd
from IPython.display import display

from zevm import fleet, plots, scenarios
from zevm.scenarios import Scenario

plots.use_style()
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.3f}".format)

## History and validation

The parc starts from the DfT year-end count in the base year. Fuel use per mile is calibrated from DESNZ fuel consumption
(`calibration_years`, lockdown years left out). CO2e from that fuel is compared with the NAEI inventory for cars (1A3bi),
and scaled by their ratio over the calibration years.

In [ ]:
fp = fleet.load_params()
hist = fleet.load_history(fp)
cal = fleet.calibration(hist, fp)
val = fleet.validate(hist, fp)
print(f"base year {hist.base_year}; UK/GB registrations ratio {hist.uk_gb_ratio:.3f}")
print(f"calibration years {cal['calibration_years']}; NAEI scale {cal['co2_scale']:.3f}")
val[["co2_total_mt", "naei_mt", "ratio"]]

Backtest of the stock model: start from the actual 2014 parc, add actual DfT registrations, and compare with the actual parc.
Diesel drifts high (real diesels left the road faster than the fixed lifespan implies); petrol stays within a few percent.

In [ ]:
fleet.backtest(hist, fp, 2014).pivot(index="year", columns="fuel", values="error").style.format("{:+.1%}")

## Scenarios

**Edit `SCENARIOS` to change them** (options as in `notebooks/scenarios.ipynb`). Scenarios that stop in 2030 are carried to
`EXTEND_TO` at their last market size and ZEV share.

In [ ]:
EXTEND_TO = 2035
SCENARIOS = [
    Scenario("A. Indicative target to 2035", params="scenarios/indicative_to_2035"),
    Scenario("B. 5pp below target", params="scenarios/indicative_to_2035", zev_share_vs_target=-0.05),
    Scenario("C. Law to 2030, then 80% held"),
]
h = scenarios.prepare("car")
summary = scenarios.market_summary(scenarios.run_many(h, SCENARIOS))
regs = fleet.registrations_from_summary(summary, hist, fp, extend_to=EXTEND_TO)
out = fleet.run_many(hist, regs, fp)
tot = fleet.totals(out)
tot[tot.basis == "projected"].pivot(index="year", columns="scenario", values="co2_mt")

In [ ]:
plots.emissions_vs_naei(val, tot, cal["co2_scale"]);

In [ ]:
plots.scenario_lines(tot[tot.basis == "projected"], "bev_share_of_parc", "BEV share of cars on the road", pct=True);

In [ ]:
for name in [SCENARIOS[0].name, SCENARIOS[1].name]:
    plots.parc_by_fuel(out, name)

In [ ]:
plots.parc_by_fuel(out, SCENARIOS[1].name, column="co2_mt");

Cumulative tailpipe CO2e over the projection, and the difference from scenario A.

In [ ]:
cum = tot[tot.basis == "projected"].groupby("scenario")["co2_mt"].sum().rename("Mt CO2e, cumulative")
pd.DataFrame({"cumulative_mt": cum, "vs_A_mt": cum - cum.loc[SCENARIOS[0].name]})

## Export

In [ ]:
from pathlib import Path
o = Path("../outputs"); o.mkdir(exist_ok=True)
out.to_csv(o / f"fleet_car_{h.as_of:%Y-%m}.csv", index=False)
tot.to_csv(o / f"fleet_car_totals_{h.as_of:%Y-%m}.csv", index=False)